In [3]:
%pip install unsloth transformers datasets trl peft accelerate bitsandbytes

  Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached numpy-2.5.1-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached tyro-1.0.15-py3-none-any.whl.metadata (12 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached xformers-0.0.35-py39-none-win_amd64.whl.metadata (1.0 kB)
  Using cached bitsandbytes-0.50.0-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached huggingface_hub-1.26.0-py3-none-any.whl.metadata (16 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-win_amd64.whl.metadata (1.8 kB)
  Using cached diffusers-0.39.0-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.met


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [12]:
import torch

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Torch version: 2.11.0+cpu
CUDA version: None
CUDA available: False
GPU: None


In [11]:
%pip show torch

Name: torch
Version: 2.11.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: d:\Projects\Healthcare Scheduling System\.venv\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, bitsandbytes, cut-cross-entropy, peft, torchvision, unsloth, unsloth_zoo, xformers
Note: you may need to restart the kernel to use updated packages.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train"
)

In [ ]:
def format_dolly(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}

dataset = ds.map(format_dolly)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,          # Reduce from 2048
    packing=True,                 # Enable sequence packing
    args=TrainingArguments(
        output_dir="outputs",

        num_train_epochs=2,       # 2 epochs are usually enough

        learning_rate=1e-4,

        per_device_train_batch_size=4,   # Increase if VRAM allows
        gradient_accumulation_steps=2,   # Keep effective batch size = 8

        logging_steps=10,

        save_strategy="epoch",

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        optim="adamw_8bit",

        report_to="none",

        lr_scheduler_type="cosine",
        warmup_ratio=0.03,

        weight_decay=0.01,
    ),
)


In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("phi3-dolly-lora")
tokenizer.save_pretrained("phi3-dolly-lora")

In [ ]:
model.save_pretrained_gguf(
    "gguf_model",
    tokenizer,
    quantization_method="q4_k_m",
)